In [1]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid
from datetime import date

task_id = 50
parent_run_id = ""
task_executions_id = ""
lineage_id = str(uuid.uuid4())
source_connection_settings = "{}"
limit_rows_for_debugging = False

# Manual fallback for source settings.
source_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "folder_path": "Files/silver/silver_transactional_csv",
    "source_system": "lh_canada_global_mds",
    "table_config": [
        {"source_file": "EFI.csv", "target_table": "efi"},
        {"source_file": "TME.csv", "target_table": "tme"},
        {"source_file": "ChronoMetrics.csv", "target_table": "chrono_metrics"}
    ]
})

# Manual fallback for target settings.
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_transactional",
    "load_strategy": "overwrite"
})

StatementMeta(, 7160143f-94d4-4037-819c-30325899adf6, 3, Finished, Available, Finished, False)

In [2]:
%run nb_transactional_shared_functions

StatementMeta(, 7160143f-94d4-4037-819c-30325899adf6, 8, Finished, Available, Finished, True)

In [3]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 7160143f-94d4-4037-819c-30325899adf6, 9, Finished, Available, Finished, False)

In [4]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_lakehouse_name = source_settings.get("lakehouse_name")
source_folder_path = source_settings.get("folder_path")
source_system = source_settings.get("source_system", source_lakehouse_name)
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "overwrite")

# Resolve lakehouse storage paths by name.
source_lakehouse_path = notebookutils.lakehouse.get(source_lakehouse_name).properties["abfsPath"]
target_lakehouse_path = notebookutils.lakehouse.get(target_lakehouse_name).properties["abfsPath"]

StatementMeta(, 7160143f-94d4-4037-819c-30325899adf6, 10, Finished, Available, Finished, False)

In [ ]:
# Load csv to delta
status = "Failed"

try:
    for cfg in table_config:
        source_file = cfg["source_file"]
        target_table = cfg["target_table"]

        # Build source file path and target delta table path.
        source_file_path = f"{source_lakehouse_path}/{source_folder_path}/{source_file}"
        target_table_path = f"{target_lakehouse_path}/Tables/{target_schema}/{target_table}"

        logger.info(f"Processing {source_file} -> {target_schema}.{target_table}")
        logger.info(f"Source path: {source_file_path}")
        logger.info(f"Target path: {target_table_path}")

        # Read source CSV file.
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("delimiter", ",")
            .load(source_file_path)
        )

        # Apply framework-aligned cleanup.
        df = clean_bronze_table(df, limit_rows_for_debugging)

        # Remove exact duplicate rows.
        before_count = df.count()
        df = remove_exact_duplicates(df)
        after_count = df.count()

        logger.info(f"{source_file} removed {before_count - after_count} duplicate rows")

        # Stamp framework metadata columns.
        df = add_metadata(
            df=df,
            ingest_date=str(date.today()),
            file_path=source_file_path,
            schema_name=source_system,
            table_name=target_table,
            lineage_id=lineage_id
        )

        # Write to the target table path.
        # For this base notebook we only care about overwrite/append.
        load_to_target(df, target_table_path, target_load_strategy)

        logger.info(f"Loaded {source_file} into {target_table_path}")

    status = "Completed"

except Exception:
    logger.exception("Processing error.")
    status = "Failed"
    raise

In [ ]:
# Read back from the target paths
for cfg in table_config:
    target_table = cfg["target_table"]
    target_table_path = f"{target_lakehouse_path}/Tables/{target_schema}/{target_table}"

    df_check = spark.read.format("delta").load(target_table_path)
    print(f"{target_schema}.{target_table}: {df_check.count()} rows")
    display(df_check.limit(5))

In [ ]:
# Task validation
result = {
    "status": status,
    "task_id": task_id,
    "task_executions_id": task_executions_id,
    "lineage_id": lineage_id,
    "source_lakehouse_name": source_lakehouse_name,
    "target_lakehouse_name": target_lakehouse_name,
    "target_schema": target_schema,
    "tables_processed": [cfg["target_table"] for cfg in table_config]
}

print(result)